Import all necessary libraries, covering parsing logs,labeling, analysis, recons

In [1]:
import pandas as pd
from pathlib import Path

# -------- parsing ----------
from parsing.syslog import parse_syslog_csv
from parsing.auditd import parse_audit_log
from parsing.auth import parse_auth_log
from parsing.zeek import (
    parse_conn_log, parse_dns_log, parse_http_log, 
    parse_files_log, parse_ssl_log, parse_ssh_log
)
from parsing.suricata import parse_suricata_eve
from parsing.azure_wins import (
    parse_azure_conn, parse_azure_process, parse_azure_security,
    parse_azure_events, parse_azure_port, parse_azure_perf
)
from parsing.tracee import parse_tracee_ndjson

# ---------- labeling / analysis / recons ----------
from analysis.coverage import step_coverage
from analysis.failure_taxonomy import failure_taxonomy 
from analysis.metrics import compute_metrics
from analysis.ambig import ambiguity

from recons.event_graph import build_event_graph
from recons.chain_builder import reconstruct_chain_detailed
# --- scenarios config ---
from scenarios import SCENARIOS

import json

# expected-step scoring from ATT&CK technique layers
try:
    from labeling.rules import expected_steps_from_techniques
except Exception:
    from rules import expected_steps_from_techniques

import math

# ----- numeric rounding helper (keep 4 decimal places during calculations) -----
def r4(x):
    """Round numeric to 4 decimals; keep None/NaN unchanged."""
    if x is None:
        return None
    try:
        xf = float(x)
    except Exception:
        return x
    try:
        if math.isnan(xf):
            return xf
    except Exception:
        pass
    return round(xf, 4)


In [2]:
import inspect
import numpy as np
import pandas as pd
import ipaddress
from labeling.tagger import tag_steps as tag_steps_base
from labeling.tagger import _is_public_ip

def _apply_azure_conn_rules(tagged: pd.DataFrame, allowed_steps=None):
    """
    Conservative heuristics for azure_conn to rescue SC1:
      - DOWNLOAD: bytes_in large and dominates bytes_out
      - EXFIL:    bytes_out large and dominates bytes_in
      - OUTBOUND_CONN: public dst + common ports (+ outbound direction if present)
    Only fills when step is empty; never overwrites.
    """
    df = tagged.copy()
    if len(df) == 0:
        return df, {"rules": {}}

    if "step" not in df.columns:
        df["step"] = pd.NA

    if "source" not in df.columns:
        return df, {"rules": {}}

    # gate steps (optional)
    allowed_set = None
    if allowed_steps is not None:
        allowed_set = {str(s).strip().upper() for s in allowed_steps if str(s).strip()}

    mask = (df["source"].astype(str) == "azure_conn")
    sub = df.loc[mask].copy()
    if len(sub) == 0:
        return df, {"rules": {}}

    need_cols = {"dst_ip", "dst_port", "bytes_in", "bytes_out"}
    missing = [c for c in need_cols if c not in sub.columns]
    if missing:
        return df, {"rules": {"QF_AZURE_CONN_SKIPPED": {"hits": 0, "missing_fields": missing}}}

    dst_ip = sub["dst_ip"].fillna("").astype(str).to_numpy()
    dst_port = sub["dst_port"].fillna("").astype(str).to_numpy()

    have_dst = dst_ip != ""
    have_port = dst_port != ""
    is_pub = np.array([_is_public_ip(x) for x in dst_ip], dtype=bool)

    # optional direction constraint
    if "direction" in sub.columns:
        direc = sub["direction"].fillna("").astype(str).str.lower().to_numpy()
        outbound_dir = np.array([("out" in x) or ("egress" in x) for x in direc], dtype=bool)
    else:
        outbound_dir = np.ones(len(sub), dtype=bool)

    ports_ok = np.isin(dst_port, np.array(["22", "80", "443", "8080", "8443"]))

    bi = pd.to_numeric(sub["bytes_in"], errors="coerce").fillna(0).astype("int64").to_numpy()
    bo = pd.to_numeric(sub["bytes_out"], errors="coerce").fillna(0).astype("int64").to_numpy()

    one_mb = 1 * 1024 * 1024
    download = (bi >= one_mb) & (bi >= 3 * np.maximum(bo, 1))
    exfil    = (bo >= one_mb) & (bo >= 3 * np.maximum(bi, 1))
    outbound = have_dst & have_port & is_pub & outbound_dir & ports_ok

    step_empty = (sub["step"].isna() | (sub["step"].astype(str).str.strip() == "")).to_numpy()

    # fill indices (never overwrite)
    fill_exfil_idx   = sub.index[step_empty & exfil]
    fill_dl_idx      = sub.index[step_empty & (~exfil) & download]         # exfil 优先于 download
    fill_outbound_idx= sub.index[step_empty & (~exfil) & (~download) & outbound]

    # apply gates
    if allowed_set is None or "EXFIL" in allowed_set:
        df.loc[fill_exfil_idx, "step"] = "EXFIL"
    else:
        fill_exfil_idx = []

    if allowed_set is None or "DOWNLOAD" in allowed_set:
        df.loc[fill_dl_idx, "step"] = "DOWNLOAD"
    else:
        fill_dl_idx = []

    if allowed_set is None or "OUTBOUND_CONN" in allowed_set:
        df.loc[fill_outbound_idx, "step"] = "OUTBOUND_CONN"
    else:
        fill_outbound_idx = []

    stats = {
        "rules": {
            "QF_AZURE_EXFIL": {"hits": int(len(fill_exfil_idx))},
            "QF_AZURE_DOWNLOAD": {"hits": int(len(fill_dl_idx))},
            "QF_AZURE_OUTBOUND_CONN": {"hits": int(len(fill_outbound_idx))},
        }
    }
    return df, stats

def tag_steps(
    events: pd.DataFrame,
    return_stats: bool = True,
    allowed_steps=None,
    keep_candidates: bool = True,
    debug: bool = False
):
    """
    Wrapper:
      1) Call labeling.tagger.tag_steps (new version supports allowed_steps gating)
      2) Apply azure_conn heuristics for SC1-like cases (still respects allowed_steps)
    """
    # call base tagger with compatibility (in case signature differs)
    kwargs = {"return_stats": True, "keep_candidates": keep_candidates, "debug": debug}
    sig = inspect.signature(tag_steps_base)
    if "allowed_steps" in sig.parameters:
        kwargs["allowed_steps"] = allowed_steps

    base = tag_steps_base(events, **kwargs)
    tagged, stats = base if isinstance(base, tuple) else (base, {})

    tagged2, add_stats = _apply_azure_conn_rules(tagged, allowed_steps=allowed_steps)

    if return_stats:
        stats = stats or {}
        stats.setdefault("rules", {})
        stats["rules"].update(add_stats.get("rules", {}))
        return tagged2, stats
    return tagged2


define the mapping from source -> parser

In [3]:
from typing import Callable, List, Tuple, Dict

def _find_files(log_root: Path, patterns: List[str]) -> List[Path]:
    ''' recursively find files matching any of the given patterns in log_root

    '''
    hits = []
    for pat in patterns:
        hits.extend(log_root.rglob(pat) if not pat.startswith("**/") else log_root.rglob(pat[3:]))
    # remove duplicates while preserving order
    uniq = sorted(set([p for p in hits if p.is_file()]))
    return uniq


def _parse_many(files: List[Path], parser: Callable[[str], pd.DataFrame], *, source: str, source_kind: str) -> pd.DataFrame:
    ''' Multiple files are parsed and merged using the same parser, and source fields are added.
    
    '''
    frames = []
    for p in files:
        try:
            df = parser(str(p))
        except Exception as e:
            print(f"Warning: failed to parse {p} with {parser.__name__}")
            print("The reason was:", e)
            continue
        if df is None or len(df) == 0:
            continue
        df = df.copy()
        df["source"] = source
        df["source_kind"] = source_kind
        df["source_file"] = str(p)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _parse_prefixed_components(
    log_root: Path,
    *,
    prefix: str, # "zeek"/"azure"
    component_parsers: Dict[str, Callable[[str], pd.DataFrame]],
    source: str,
) -> pd.DataFrame:
    '''
    recursively find and parse log files for components with given prefix
    '''
    frames = []
    items = component_parsers.items() if isinstance(component_parsers, dict) else component_parsers

    for comp, fn in items:
        patterns = [
            f"{prefix}_{comp}.csv",
            f"{prefix}_{comp}_*.csv",
            f"{prefix}_{comp}*.csv",
        ]
        files = _find_files(log_root, patterns)
        df = _parse_many(files, fn, source=source, source_kind=f"{source}_{comp}")
        if len(df):
            frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def parse_source(source_key: str, log_root: Path) -> pd.DataFrame:
    """
    return unified source dataframe for a given source_key
    """
    # --- syslog ---
    if source_key == "syslog":
        files = _find_files(log_root, ["azure_syslog.csv", "azure_syslog*.csv"])
        return _parse_many(files, parse_syslog_csv, source="syslog", source_kind="syslog")
    
    # --- auditd ---
    if source_key == "auditd":
        files = _find_files(log_root, ["audit.log", "auditd*.log"])
        return _parse_many(files, parse_audit_log, source="auditd", source_kind="auditd")

    # --- auth ---
    if source_key == "auth":
        files = _find_files(log_root, ["auth.log", "auth*.log"])
        return _parse_many(files, parse_auth_log, source="auth", source_kind="auth")

    # ----- suricata -----
    if source_key == "suricata":
        files = _find_files(log_root, ["eve.json", "eve*.json"])
        return _parse_many(files, parse_suricata_eve, source="suricata", source_kind="suricata")

    # ----- tracee (FIX) -----
    if source_key == "tracee":
        files = _find_files(log_root, ["tracee-events.json", "tracee*.json", "tracee*.ndjson"])
        return _parse_many(files, parse_tracee_ndjson, source="tracee", source_kind="tracee")

    # --- zeek (multi-log) ---
    if source_key == "zeek":
        zeek_parsers: List[Tuple[str, Callable[[str], pd.DataFrame]]] = [
            ("conn",  parse_conn_log),
            ("dns",   parse_dns_log),
            ("http",  parse_http_log),
            ("files", parse_files_log),
            ("ssh",   parse_ssh_log),
            ("ssl",   parse_ssl_log),
        ]
        return _parse_prefixed_components(
            log_root,
            prefix="zeek",
            component_parsers=zeek_parsers,
            source="zeek",
        )

    # --- azure windows (ADD perf) ---
    if source_key.startswith("azure_"):
        comp = source_key.replace("azure_", "", 1)  # conn/process/security/events/port/syslog/perf
        azure_component_parsers = {
            "conn":     parse_azure_conn,
            "process":  parse_azure_process,
            "security": parse_azure_security,
            "events":   parse_azure_events,
            "port":     parse_azure_port,
            "perf":     parse_azure_perf,      # <-- ADD
            "syslog":   parse_syslog_csv,
        }
        fn = azure_component_parsers.get(comp)
        if fn is None:
            return pd.DataFrame()

        patterns = [
            f"azure_{comp}.csv",
            f"azure_{comp}_*.csv",
            f"azure_{comp}*.csv",
        ]
        files = _find_files(log_root, patterns)
        return _parse_many(files, fn, source=source_key, source_kind=source_key)

    return pd.DataFrame()

run pipeline (tag + metrics + chain) with given sources

In [4]:
def _normalize_ts(df: pd.DataFrame, ts_col="ts") -> pd.DataFrame:
    if df is None or len(df) == 0 or ts_col not in df.columns:
        return df
    out = df.copy()

    ts = pd.to_datetime(out[ts_col], errors="coerce", utc=True)

    out[ts_col] = ts.dt.tz_convert(None)
    return out


def run_pipeline_for_sources(enabled_sources, log_root: Path, expected_steps=None, expected_order=None):
    """Run parse->tag->analyze->reconstruct for a given enabled source set.

    expected_steps:
      Optional list of coarse steps (e.g., ["AUTH","DOWNLOAD"]) derived from ATT&CK techniques.
      If provided, reconstruction quality is scored as step-level recall/precision against this set.
    """
    # 1) parsing
    frames = []

    for s in enabled_sources:
        df = parse_source(s, log_root)
        if df is not None and len(df) > 0:
            df["source"] = s
            frames.append(df)

    events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # --- guard: prevent cross-scenario leakage ---
    if len(events) and "source_file" in events.columns:
        root = str(Path(log_root).resolve())
        events = events[events["source_file"].astype(str).str.startswith(root)].copy()

    # 2) tagging (with schema diagnostics)
    allowed = expected_steps if expected_steps else None

    if len(events):
        tagged, tag_stats = tag_steps(events, return_stats=True, allowed_steps=allowed)
    else:
        tagged, tag_stats = events, {}

    # print(tagged.columns.tolist())

    if len(tagged) and "ts" in tagged.columns:
        tagged = tagged.copy()
        tagged["ts"] = pd.to_datetime(tagged["ts"], utc=True, errors="coerce").dt.tz_convert(None)

    # 3) analytics
    cov = step_coverage(tagged) if len(tagged) else {}
    fails = failure_taxonomy(tagged) if len(tagged) else {}
    amb = ambiguity(tagged) if len(tagged) else {}

    # 4) reconstruction (ANCHOR by expected_order)
    tagged_for_recon = tagged
    if len(tagged) and expected_order and "step" in tagged.columns and "ts" in tagged.columns:
        present = set(tagged["step"].dropna().unique().tolist())
        anchor_candidates = [s for s in expected_order if s in present]
        if anchor_candidates:
            anchor_step = anchor_candidates[0]
            t0 = tagged.loc[tagged["step"] == anchor_step, "ts"].min()
            if pd.notna(t0):
                tagged_for_recon = tagged[tagged["ts"] >= (t0 - pd.Timedelta(seconds=120))].copy()

    g = None
    chain_steps = []
    if len(tagged_for_recon) and "step" in tagged_for_recon.columns and "ts" in tagged_for_recon.columns:
        g = build_event_graph(tagged_for_recon)
        d = reconstruct_chain_detailed(g, tagged_for_recon.dropna(subset=["step"]))
        chain_steps = d.get("steps", [])

    # 5) metrics (chain-aware)
    metrics = compute_metrics(tagged, chain=chain_steps, max_gap_seconds=600) if len(tagged) else {}

    # 6) reconstruction summary
    observed_steps = []
    if len(tagged) and "step" in tagged.columns:
        observed_steps = sorted(tagged["step"].dropna().unique().tolist())

    expected_steps = expected_steps or []
    expected_set = set(expected_steps)

    # display order: expected_order first, then the rest (do NOT change scoring semantics)
    eo = expected_order or []
    expected_steps_display = [s for s in eo if s in expected_set] + sorted(list(expected_set - set(eo)))

    observed_set = set(observed_steps)
    chain_set = set(chain_steps)

    step_recall = r4((len(observed_set & expected_set) / len(expected_set)) if expected_set else None)
    step_precision = r4((len(observed_set & expected_set) / len(observed_set)) if observed_set else None)

    chain_recall = r4((len(chain_set & expected_set) / len(expected_set)) if expected_set else None)
    chain_precision = r4((len(chain_set & expected_set) / len(chain_set)) if chain_set else None)

    # Reconstructability:
    # - primary metric: expected-step recall (when scenario ground truth is available)
    # - otherwise: leave reconstructability as None/NaN and provide a proxy for exploratory analysis only
    if expected_set:
        reconstructability = step_recall
        reconstructability_mode = "expected_recall"
        proxy_reconstructability = None
    else:
        reconstructability = None
        reconstructability_mode = "proxy_chain_over_observed"
        proxy_reconstructability = r4((len(chain_steps) / len(observed_steps)) if observed_steps else 0.0)

    # compact tagger diagnostics (helps explain silent misses)
    diag = {
        "tagger_total_hits": 0,
        "tagger_missing_fields_events": 0,
        "tagger_missing_prefilter_events": 0,
        "tagger_used_prefilter_conditions": 0,
        "tagger_unusable_rules": [],
    }
    if isinstance(tag_stats, dict) and "rules" in tag_stats:
        for rid, rs in tag_stats["rules"].items():
            diag["tagger_total_hits"] += int(rs.get("hits", 0) or 0)
            diag["tagger_missing_fields_events"] += int(rs.get("missing_fields_events", 0) or 0)
            diag["tagger_missing_prefilter_events"] += int(rs.get("missing_prefilter_events", 0) or 0)
            diag["tagger_used_prefilter_conditions"] += int(rs.get("used_prefilter_conditions", 0) or 0)
            if (rs.get("hits", 0) or 0) == 0 and (rs.get("missing_fields_events", 0) or 0) > 0:
                diag["tagger_unusable_rules"].append(rid)

    recon = {
        "n_events": int(len(events)) if len(events) else 0,
        "expected_steps": expected_steps_display,       
        "expected_steps_set": sorted(list(expected_set)), 
        "n_expected_steps": int(len(expected_set)),
        "expected_order": eo, 
        "observed_steps": observed_steps,
        "n_tagged_steps": int(len(observed_steps)),
        "chain_steps": chain_steps,
        "n_chain_steps": int(len(chain_steps)),
        "reconstructability": reconstructability,
        "reconstructability_mode": reconstructability_mode,
        "proxy_reconstructability": proxy_reconstructability,
        "step_recall": step_recall,
        "step_precision": step_precision,
        "chain_recall": chain_recall,
        "chain_precision": chain_precision,
        **diag,
    }

    return {
        "events": events,
        "tagged": tagged,
        "coverage": cov,
        "failures": fails,
        "ambiguity": amb,
        "metrics": metrics,
        "chain": chain_steps,
        "graph": g,
        "recon": recon,
    }


### comparison between single source vs multiple sources

- multiple sources: all sources from one scenario
- single source: every source in one scenarios separately runs once, find failure mode and missing part
- output one summary

In [5]:
import re
import pandas as pd
from collections import defaultdict

TECH_RULES = {
    # --- Remote Services / Credential Access ---
    "T1021.004": {  # Remote Services: SSH
        "patterns": [r"\bsshd\b", r"\bssh2\b", r"Accepted .* ssh", r"Failed password"],
        "confidence": "medium",
        "evidence_level": "auth_log_or_syslog",
    },
    "T1021.001": {  # Remote Services: RDP (xrdp)
        "patterns": [r"\bxrdp\b", r"\bxrdp-sesman\b", r"\brdp\b.*(connect|login)"],
        "confidence": "low",
        "evidence_level": "syslog_process_or_message",
    },
    "T1110": {  # Brute Force
        "patterns": [r"Failed password", r"authentication failure", r"invalid user"],
        "confidence": "high",
        "evidence_level": "auth_log",
    },

    # --- Persistence / Privilege Escalation ---
    "T1053.003": {  # Scheduled Task/Job: Cron
        "patterns": [
            r"\bCRON\b", r"cron:session", r"/usr/sbin/CRON",
            r"COMMAND=.*\bcrontab\b", r"COMMAND=.*\b/etc/cron\.", r"@reboot"
        ],
        "confidence": "medium",
        "evidence_level": "syslog_or_sudo_command",
    },
    "T1548.003": {  # Abuse Elevation Control Mechanism: Sudo and Sudo Caching
        "patterns": [
            r"pam_unix\(sudo:session\): session opened",
            r"pam_unix\(sudo:session\): session closed",
            r"\bsudo\b"
        ],
        "confidence": "medium",
        "evidence_level": "auth_log_or_syslog",
    },
    "T1543.002": {  # Create or Modify System Process: Systemd Service
        "patterns": [
            r"COMMAND=.*\bsystemctl\b.*\b(enable|disable|mask|unmask|daemon-reload|start|stop|restart)\b",
            r"/etc/systemd/system/.*\.service",
            r"systemd\[1\]: .* (Started|Stopped|Reloaded)"
        ],
        "confidence": "low",
        "evidence_level": "syslog_or_sudo_command",
    },

    # --- Defense Evasion (impair defenses / logs) ---
    "T1562.001": {  # Impair Defenses: Disable or Modify Tools
        "patterns": [
            r"COMMAND=.*\bsystemctl\b.*\b(stop|disable|mask)\b.*\b(auditd|rsyslog|osqueryd|falcon-sensor|clamav|ufw)\b",
            r"\bauditctl\b.*\s-e\s*0\b",
            r"COMMAND=.*\bsetenforce\b\s+0\b"
        ],
        "confidence": "medium",
        "evidence_level": "suspicious_admin_action",
    },
    "T1562.004": {  # Impair Defenses: Disable or Modify System Firewall
        "patterns": [
            r"COMMAND=.*\bufw\b.*\b(disable|reset)\b",
            r"COMMAND=.*\biptables\b.*\s(-F|--flush)\b",
            r"COMMAND=.*\bfirewall-cmd\b.*\b(--permanent|--remove|--add)\b"
        ],
        "confidence": "medium",
        "evidence_level": "firewall_change_cmd",
    },
    "T1070.003": {  # Indicator Removal: Clear Command History
        "patterns": [
            r"COMMAND=.*\bhistory\b\s+-c\b",
            r"COMMAND=.*\brm\b.*\b\.bash_history\b",
            r"COMMAND=.*\bunset\b\s+HISTFILE\b"
        ],
        "confidence": "medium",
        "evidence_level": "shell_history_tamper",
    },
    "T1070.004": {  # Indicator Removal: File Deletion
        "patterns": [
            r"COMMAND=.*\brm\b\s+-rf\b",
            r"COMMAND=.*\bshred\b",
            r"COMMAND=.*\btruncate\b",
            r"COMMAND=.*>\s*/var/log/"
        ],
        "confidence": "low",
        "evidence_level": "destructive_cmd",
    },

    # --- Command & Scripting Interpreter ---
    "T1059.004": {  # Command and Scripting Interpreter: Unix Shell
        "patterns": [
            r"COMMAND=.*\b(bash|sh|zsh|dash|ksh)\b",
            r"proctitle_decoded=.*\b(bash|sh|zsh|dash|ksh)\b"
        ],
        "confidence": "low",
        "evidence_level": "cmdline_or_audit_proctitle",
    },
    "T1059.006": {  # Python
        "patterns": [
            r"COMMAND=.*\bpython3?\b",
            r"proctitle_decoded=.*\bpython3?\b"
        ],
        "confidence": "low",
        "evidence_level": "cmdline_or_audit_proctitle",
    },

    # --- Ingress Tool Transfer / Download ---
    "T1105": {  # Ingress Tool Transfer
        "patterns": [
            r"COMMAND=.*\b(curl|wget)\b",
            r"proctitle_decoded=.*\b(curl|wget)\b"
        ],
        "confidence": "medium",
        "evidence_level": "download_cmd",
    },

    # --- Discovery (very observable via commands) ---
    "T1082": {  # System Information Discovery
        "patterns": [
            r"COMMAND=.*\b(uname|hostnamectl|lsb_release)\b",
            r"COMMAND=.*\bcat\b\s+/etc/os-release\b"
        ],
        "confidence": "low",
        "evidence_level": "discovery_cmd",
    },
    "T1016": {  # System Network Configuration Discovery
        "patterns": [
            r"COMMAND=.*\b(ip\s+a|ip\s+addr|ip\s+route|ifconfig|route\s+-n|nmcli)\b",
            r"COMMAND=.*\b(cat|less)\b\s+/etc/resolv\.conf\b"
        ],
        "confidence": "low",
        "evidence_level": "discovery_cmd",
    },
    "T1057": {  # Process Discovery
        "patterns": [
            r"COMMAND=.*\b(ps\s+aux|ps\s+-ef|pgrep|top)\b"
        ],
        "confidence": "low",
        "evidence_level": "discovery_cmd",
    },
    "T1033": {  # System Owner/User Discovery
        "patterns": [
            r"COMMAND=.*\b(whoami|id|who|w|last)\b"
        ],
        "confidence": "low",
        "evidence_level": "discovery_cmd",
    },
    "T1083": {  # File and Directory Discovery
        "patterns": [
            r"COMMAND=.*\b(find|locate|tree|ls\s+-la)\b"
        ],
        "confidence": "low",
        "evidence_level": "discovery_cmd",
    },
    "T1046": {  # Network Service Scanning
        "patterns": [
            r"COMMAND=.*\b(nmap|masscan)\b"
        ],
        "confidence": "medium",
        "evidence_level": "scan_cmd",
    },

    # --- Accounts ---
    "T1136.001": {  # Create Account: Local Account
        "patterns": [
            r"COMMAND=.*\b(useradd|adduser)\b",
            r"COMMAND=.*\bgroupadd\b"
        ],
        "confidence": "medium",
        "evidence_level": "account_admin_cmd",
    },
    "T1098": {  # Account Manipulation (generic)
        "patterns": [
            r"COMMAND=.*\b(usermod|userdel|passwd|chpasswd)\b"
        ],
        "confidence": "low",
        "evidence_level": "account_admin_cmd",
    },
    "T1098.004": {  # SSH Authorized Keys
        "patterns": [
            r"COMMAND=.*authorized_keys\b",
            r"proctitle_decoded=.*authorized_keys\b",
            r"Accepted publickey"
        ],
        "confidence": "medium",
        "evidence_level": "ssh_key_change_or_login",
    },

    # --- Collection / Staging ---
    "T1560": {  # Archive Collected Data
        "patterns": [
            r"COMMAND=.*\b(tar|zip|7z|gzip)\b",
            r"proctitle_decoded=.*\b(tar|zip|7z|gzip)\b"
        ],
        "confidence": "low",
        "evidence_level": "archive_cmd",
    },
}

def _row_text(row: pd.Series, cols: list[str]) -> str:
    parts = []
    for c in cols:
        if c in row and pd.notna(row[c]):
            parts.append(str(row[c]))
    return " | ".join(parts)

def extract_observed_ttps(df: pd.DataFrame,
                          text_cols=("cmdline","command","process","proc","exe","SyslogMessage","message","raw","event","proctitle"),
                          source_col=("source","Source","log","Type","dataset")) -> pd.DataFrame:
    """
    input：pipeline output（DataFrame)
    output：observed technique evidence table（technique_id, hit_count, sources, evidence_level, confidence, samples）
    """
    text_cols = [c for c in text_cols if c in df.columns]

    use_all_cols = False
    if not text_cols:
        use_all_cols = True

    src_col = next((c for c in source_col if c in df.columns), None)

    agg = defaultdict(lambda: {"hit_count":0, "sources":set(), "samples":[]})
    for _, row in df.iterrows():
        if use_all_cols:
            txt = " | ".join(str(v) for v in row.values if v is not None)
        else:
            txt = _row_text(row, text_cols)

        if not txt:
            continue

        for tid, rule in TECH_RULES.items():
            if any(re.search(p, txt, flags=re.IGNORECASE) for p in rule["patterns"]):
                agg[tid]["hit_count"] += 1
                if src_col:
                    agg[tid]["sources"].add(str(row[src_col]))
                if len(agg[tid]["samples"]) < 3:
                    agg[tid]["samples"].append(txt[:220])

    out = []
    for tid, v in agg.items():
        out.append({
            "technique_id": tid,
            "hit_count": v["hit_count"],
            "sources": ";".join(sorted(v["sources"])) if v["sources"] else "",
            "evidence_level": TECH_RULES[tid]["evidence_level"],
            "confidence": TECH_RULES[tid]["confidence"],
            "samples": " || ".join(v["samples"]),
        })
    
    CONF_RANK = {"low": 0, "medium": 1, "high": 2}

    out_df = pd.DataFrame(out)
    if out_df.empty:
        return pd.DataFrame(columns=[
            "technique_id","hit_count","sources","evidence_level","confidence","samples"
        ])

    out_df["confidence_rank"] = out_df["confidence"].map(CONF_RANK).fillna(0).astype(int)

    out_df = out_df.sort_values(["confidence_rank","hit_count"], ascending=[False, False])

    return out_df.drop(columns=["confidence_rank"]).reset_index(drop=True)


In [6]:
def get_expected_sources_for_scenario(sc) -> list:
    # sc.expected_sources is dict[str,bool]
    return [k for k, v in sc.expected_sources.items() if v]


def _load_layer_techniques(layer_path: Path) -> list:
    """Load techniqueIDs from an ATT&CK Navigator layer JSON."""
    try:
        j = json.loads(layer_path.read_text())
    except Exception:
        return []

    techs = []
    for t in j.get("techniques", []) or []:
        tid = t.get("techniqueID")
        if tid:
            techs.append(tid)
    return techs


def load_scenario_ttps(log_root: Path) -> list:
    """Best-effort discovery of ATT&CK layer files for a scenario.

    Order:
      1) scenario-local *ttps*.json / attack_navigator_*.json
      2) fallback to repository-level defaults if present
    """
    log_root = Path(log_root)

    layer_files = []
    layer_files += sorted(log_root.glob("*ttp*.json"))
    layer_files += sorted(log_root.glob("*ttps*.json"))
    layer_files += sorted(log_root.glob("attack_navigator_*.json"))

    techs = []
    for f in layer_files:
        techs.extend(_load_layer_techniques(f))

    # de-dup preserve sort
    return sorted(set(techs))

EXPECTED_ORDER = {
    # NOTE: Ground-truth coarse-step order for evaluation (paper-facing).
    # Add/adjust steps here to keep the scoring definition consistent with the labeling rules.
    "SC1": ["INSTALL", "DOWNLOAD", "OUTBOUND_CONN", "EXFIL"],  # Stegano
    "SC2": ["INSTALL", "DOWNLOAD", "OUTBOUND_CONN", "EXFIL"],  # Starter
    "SC3": ["INSTALL", "DOWNLOAD", "OUTBOUND_CONN", "EXFIL"],  # Parallel
    "SC4": ["INSTALL", "DOWNLOAD", "OUTBOUND_CONN", "EXFIL"],  # NPMEX
    "SC5": ["INSTALL", "DOWNLOAD", "OUTBOUND_CONN", "EXFIL"],  # 3CX
    "SC6": ["AUTH", "DOWNLOAD", "OUTBOUND_CONN", "EXFIL"],     # CloudEX
    "SC7": ["DOWNLOAD", "OUTBOUND_CONN", "EXFIL"],             # LayerInj
}


# --- Source-budgeting helper (paper-facing) ---
# Treat azure_events as a composite telemetry stream (multiple evidence channels), so it should NOT be
# counted as a single source for budget comparisons.
COMPOSITE_SOURCE_WEIGHTS = {"azure_events": 3}  # counts as 3 sources total for budgeting

def effective_n_sources(source_set: str, weights: dict = COMPOSITE_SOURCE_WEIGHTS) -> int:
    parts = str(source_set).split("+") if source_set else []
    n = len(parts)
    for src, w in (weights or {}).items():
        if src in parts:
            n += (int(w) - 1)
    return n

def budget_mode_from_source_set(source_set: str) -> str:
    n = effective_n_sources(source_set)
    if n <= 1:
        return "single"
    if n == 2:
        return "combo2"
    return "multi"


def analyze_scenario(scenario_key: str, log_root: Path):
    log_root = Path(log_root)

    # if caller passed the global root, resolve scenario dir automatically
    cands = [
        log_root / scenario_key,
        log_root / scenario_key.lower(),
        log_root / scenario_key.upper(),
    ]
    for d in cands:
        if d.exists():
            log_root = d
            break

    # extract scenario config
    sc = SCENARIOS[scenario_key]
    expected_sources = get_expected_sources_for_scenario(sc)

    # expected steps derived from ATT&CK layer(s) (if available)
    technique_ids = load_scenario_ttps(log_root)
    expected_steps_layer = expected_steps_from_techniques(technique_ids)

    expected_order = EXPECTED_ORDER.get(sc.id, EXPECTED_ORDER.get(str(scenario_key).upper(), []))
    # Prefer manual expected_order for coarse-step evaluation; fall back to layer-derived if missing.
    expected_steps = expected_order if expected_order else expected_steps_layer

    def derive_failure_keys(run_out: dict) -> list:
        """Stable failure keys for Q2/Q3, even when heuristic failure_taxonomy is empty."""
        recon = run_out.get("recon", {}) if isinstance(run_out, dict) else {}
        failures = run_out.get("failures", {}) if isinstance(run_out, dict) else {}

        expected = set(recon.get("expected_steps") or [])
        observed = set(recon.get("observed_steps") or [])
        chain = set(recon.get("chain_steps") or [])

        keys = set()

        # ground-truth availability
        if not expected:
            keys.add("NO_EXPECTED_STEPS")
        else:
            for st in sorted(expected - observed):
                keys.add(f"MISSING_{st}")
            for st in sorted(observed - expected):
                keys.add(f"EXTRA_{st}")

        # reconstruction gaps
        if recon.get("n_events", 0) > 0 and not observed:
            keys.add("NO_STEPS_OBSERVED")
        if observed and not chain:
            keys.add("NO_CHAIN_RECONSTRUCTED")

        # schema / rule usability diagnostics
        if (recon.get("tagger_missing_fields_events") or 0) > 0:
            keys.add("SCHEMA_MISMATCH_FIELDS")
        if (recon.get("tagger_missing_prefilter_events") or 0) > 0:
            keys.add("PREFILTER_UNUSABLE")
        if (recon.get("tagger_unusable_rules") or []):
            keys.add("UNUSABLE_RULES_PRESENT")

        # include heuristic taxonomy if present
        if isinstance(failures, dict):
            keys |= set(failures.keys())

        return sorted(keys)

    # ---- multi-source run -----
    multi = run_pipeline_for_sources(
        expected_sources, log_root,
        expected_steps=expected_steps,
        expected_order=expected_order
    )

    # expected (from layers)
    expected = set(technique_ids)

    # observed (from logs)
    tagged = multi["tagged"]          # DataFrame
    observed_df = extract_observed_ttps(tagged)
    observed = set(observed_df["technique_id"].tolist())

    aligned = sorted(expected & observed)
    missing = sorted(expected - observed)
    unexpected = sorted(observed - expected)

    multi["recon"]["observed_techniques"] = sorted(observed)
    multi["recon"]["aligned_techniques"] = aligned
    multi["recon"]["missing_expected_techniques"] = missing
    multi["recon"]["unexpected_observed_techniques"] = unexpected

    multi["metrics"]["ttp_alignment_coverage"] = r4(len(aligned) / max(1, len(expected)))

    # --- single-source runs ---
    single_rows = []
    single_outputs = {}
    for s in expected_sources:
        out = run_pipeline_for_sources([s], log_root, expected_steps=expected_steps, expected_order=expected_order)
        single_outputs[s] = out
        if budget_mode_from_source_set(s) == "single":
            single_rows.append({
            "scenario": sc.id,
            "name": sc.name,
            "run_mode": "single",
            "mode": budget_mode_from_source_set(s),
            "source_set": s,
            **out["recon"],
            "failure_keys": derive_failure_keys(out),
            })

    # best single-source by recon
    single_df = pd.DataFrame(single_rows)
    if len(single_df):
        best_single = single_df.sort_values(
            ["reconstructability", "n_chain_steps", "n_tagged_steps"],
            ascending=False,
        ).head(1)
        best_single_row = best_single.iloc[0].to_dict()
    else:
        best_single_row = {}

    # --- combination runs (2 sources) ---
    # These combos are meant to align with "limited-source" related work baselines:
    #   - host+network: auditd+zeek
    #   - system+auth:  syslog+auth
    #   - system+app:   syslog+azure_events (if present)
    # Set RUN_ALL_COMBO2_PAIRS=True to also evaluate all 2-source pairs available in this scenario.
    import itertools
    RUN_ALL_COMBO2_PAIRS = False
    COMBO2_PRESETS = [
        ("auditd", "zeek"),
        ("syslog", "auth"),
        ("syslog", "azure_events"),
        ("auditd", "syslog"),
        ("zeek", "syslog"),
        ("zeek", "auth"),
        ("auditd", "azure_events"),
        ("zeek", "azure_events"),
    ]

    combo2_rows = []
    combo2_outputs = {}

    cand = list(COMBO2_PRESETS)
    if RUN_ALL_COMBO2_PAIRS:
        cand.extend(list(itertools.combinations(expected_sources, 2)))

    # de-dup + filter by availability in this scenario
    seen = set()
    pairs = []
    for a, b in cand:
        if a in expected_sources and b in expected_sources:
            key = tuple(sorted((a, b)))
            if key in seen:
                continue
            seen.add(key)
            pairs.append([a, b])

    for pair in pairs:
        out = run_pipeline_for_sources(pair, log_root, expected_steps=expected_steps, expected_order=expected_order)
        combo2_outputs["+".join(pair)] = out
        combo2_rows.append({
            "scenario": sc.id,
            "name": sc.name,
            "run_mode": "combo2",
            "mode": budget_mode_from_source_set("+".join(pair)),
            "source_set": "+".join(pair),
            **out["recon"],
            "failure_keys": derive_failure_keys(out),
        })

    combo2_df = pd.DataFrame(combo2_rows)


    # multi row
    multi_row = {
        "scenario": sc.id,
        "name": sc.name,
        "run_mode": "multi",
        "mode": budget_mode_from_source_set("+".join(expected_sources)),
        "source_set": "+".join(expected_sources),
        **multi["recon"],
        "failure_keys": sorted(list(multi["failures"].keys())) if isinstance(multi["failures"], dict) else [],
    }

    # --- Gap explanation helpers (for Q2/Q3) ---
    # Prefer expected steps (from ATT&CK) for gap explanations. If unavailable, fall back to steps observed in multi.
    expected_step_set = set(expected_steps or [])
    multi_steps = set(multi["recon"].get("observed_steps", []))

    step_presence = {}
    for s, out in single_outputs.items():
        step_presence[s] = set(out["recon"].get("observed_steps", []))

    basis_steps = sorted(expected_step_set) if expected_step_set else sorted(multi_steps)

    step_missing_in_single = {}
    for st in basis_steps:
        missing_sources = [s for s in expected_sources if st not in step_presence.get(s, set())]
        if missing_sources:
            step_missing_in_single[st] = missing_sources

    expected_steps_missing_in_multi = sorted(list(expected_step_set - multi_steps)) if expected_step_set else []

    # Which failures are most frequent in single-source scenarios but mitigated by multi-source scenarios?
    def failure_keys(x):
        return set(x.keys()) if isinstance(x, dict) else set()

    multi_fail = failure_keys(multi["failures"])
    single_fail_union = set()
    for s in expected_sources:
        single_fail_union |= failure_keys(single_outputs[s]["failures"])

    mitigated_failures = sorted(list(single_fail_union - multi_fail))

    # Score helpers (handles missing expected steps)
    def _score_from_row(row: dict) -> float:
        v = row.get("reconstructability", None)
        if v is None:
            return float(row.get("proxy_reconstructability", 0.0) or 0.0)
        try:
            # NaN check
            if v != v:
                return float(row.get("proxy_reconstructability", 0.0) or 0.0)
        except Exception:
            pass
        return float(v)

    multi_score = r4(_score_from_row(multi["recon"]))
    best_single_score = r4(_score_from_row(best_single_row)) if isinstance(best_single_row, dict) else 0.0

    # best 2-source combo (if evaluated)
    if 'combo2_df' in locals() and len(combo2_df):
        best_combo2 = combo2_df.sort_values(
            ["reconstructability", "n_chain_steps", "n_tagged_steps"],
            ascending=False,
        ).head(1)
        best_combo2_row = best_combo2.iloc[0].to_dict()
    else:
        best_combo2_row = {}

    best_combo2_score = r4(_score_from_row(best_combo2_row)) if isinstance(best_combo2_row, dict) else 0.0
    multi_minus_best_combo2 = r4(multi_score - best_combo2_score)

    multi_minus_best_single = r4(multi_score - best_single_score)

    summary = {
        "scenario": sc.id,
        "name": sc.name,
        "expected_sources": expected_sources,
        "expected_techniques": technique_ids,
        "expected_steps": expected_steps,
        "expected_steps_missing_in_multi": expected_steps_missing_in_multi,
        "multi_reconstructability": multi_score,
        "multi_reconstructability_mode": multi["recon"].get("reconstructability_mode"),
        "best_single_source": best_single_row.get("source_set"),
        "best_single_reconstructability": best_single_score,
        "multi_minus_best_single": multi_minus_best_single,
        "best_combo2_source": best_combo2_row.get("source_set"),
        "best_combo2_reconstructability": best_combo2_score,
        "multi_minus_best_combo2": multi_minus_best_combo2,
        "steps_missing_in_single": step_missing_in_single,
        "failures_mitigated_by_multi": mitigated_failures,

        # keep old keys for compatibility
        "expected_payload_ttps": technique_ids,
        "expected_c2_ttps": None,
    }

    return {
        "multi": multi,
        "single": single_outputs,
        "single_df": single_df,
        "combo2": combo2_outputs,
        "combo2_df": combo2_df,
        "multi_row": multi_row,
        "summary": summary,
    }


In [7]:
def pretty_three_questions(summary: dict) -> str:
    sc = summary["scenario"]
    name = summary["name"]
    multi_r = summary["multi_reconstructability"]
    best_s = summary["best_single_source"]
    best_r = summary["best_single_reconstructability"]
    delta = summary["multi_minus_best_single"]

    # Q1: end-to-end reconstructability
    q1 = (
        f"[Q1] {sc} ({name}) — End-to-end reconstructability (expected-step recall when available) under multi-source telemetry: "
        f"reconstructability={multi_r:.4f}. "
        f"Best single-source baseline ({best_s})={best_r:.4f} "
        f"(multi − best_single = {delta:+.4f})."
    )

    # Q2: where/why single-source fails (proxy via steps missing in single-source runs)
    missing = summary["steps_missing_in_single"]
    top_missing = list(missing.items())[:8]  # keep it readable

    if top_missing:
        q2_lines = [
            f"  - step={st} is absent in single-source runs for: {srcs}"
            for st, srcs in top_missing
        ]
        q2 = (
            "[Q2] Single-source failure points — examples of steps observable in the multi-source run "
            "but missing under individual sources:\n"
            + "\n".join(q2_lines)
        )
    else:
        q2 = (
            "[Q2] Single-source failure points — no clear step-level omissions detected "
            "(this can happen when the chain is short, when sources have overlapping visibility, "
            "or when labeling evidence is sparse)."
        )

    # Q3: how multi-source mitigates failures (proxy via failure categories that disappear in multi-source)
    mitigated = summary["failures_mitigated_by_multi"]

    if mitigated:
        q3 = (
            "[Q3] Multi-source mitigation — failure categories observed in at least one single-source run "
            "but not present in the multi-source run:\n"
            + "\n".join([f"  - {x}" for x in mitigated[:10]])
        )
    else:
        q3 = (
            "[Q3] Multi-source mitigation — no clear difference in failure keys "
            "(consider refining this with step-level failure counts and/or explicit observability-gap signals)."
        )

    return q1 + "\n" + q2 + "\n" + q3


In [8]:
LOG_ROOT = Path.cwd().parent.joinpath("sanidata")

print("LOG_ROOT:", LOG_ROOT.resolve())
print("LOG_ROOT exists:", LOG_ROOT.exists())

all_files = [p for p in LOG_ROOT.rglob("*") if p.is_file()]
print("n_files under LOG_ROOT:", len(all_files))
print("sample:", [str(p.relative_to(LOG_ROOT)) for p in all_files[:50]])


LOG_ROOT: /Users/zhuoran/Projects/SSCMDataset/sanidata
LOG_ROOT exists: True
n_files under LOG_ROOT: 121
sample: ['.DS_Store', 'sc1/attack_navigator_cmds.json', 'sc1/attack_navigator_tasks.json', 'sc1/payload_ttps_2.json', 'sc1/payload_ttps_3.json', 'sc1/.DS_Store', 'sc1/payload_ttps_0.json', 'sc1/payload_ttps_1.json', 'sc1/windows/azure_port.csv', 'sc1/windows/azure_events.parquet', 'sc1/windows/azure_process.parquet', 'sc1/windows/azure_port.parquet', 'sc1/windows/azure_conn.parquet', 'sc1/windows/azure_process.csv', 'sc1/windows/azure_security.csv', 'sc1/windows/azure_conn.csv', 'sc1/windows/azure_perf.parquet', 'sc1/windows/azure_security.parquet', 'sc1/windows/azure_perf.csv', 'sc1/windows/azure_events.csv', 'sc6/attack_navigator_cmds.json', 'sc6/attack_navigator_tasks.json', 'sc6/payload_ttps_2.json', 'sc6/payload_ttps_3.json', 'sc6/.DS_Store', 'sc6/payload_ttps_4.json', 'sc6/payload_ttps_0.json', 'sc6/payload_ttps_1.json', 'sc6/victim/azure_events.parquet', 'sc6/victim/custom_au

In [9]:

sc1_dir = LOG_ROOT / "sc1"

print("files conn:", list(sc1_dir.rglob("azure_conn.csv")))
print("files conn windows:", list(sc1_dir.rglob("windows/azure_conn.csv")))

df = parse_source("azure_conn", sc1_dir)
print("parsed rows:", len(df))
print("cols:", df.columns.tolist()[:20] if len(df) else None)

files conn: [PosixPath('/Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/azure_conn.csv')]
files conn windows: [PosixPath('/Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/azure_conn.csv')]
parsed rows: 5746
cols: ['ts', 'host', 'source', 'event_type', 'action', 'subject', 'object', 'src', 'dst', 'raw', 'source_kind', 'source_file']


In [10]:
# ---- run (scenario-local root) ----
scenario_keys = list(SCENARIOS.keys())
all_rows = []
all_summaries = {}

for k in scenario_keys:
    res = analyze_scenario(k, LOG_ROOT)  
    all_rows.append(res["multi_row"])
    if len(res["single_df"]):
        all_rows.extend(res["single_df"].to_dict("records"))
    if "combo2_df" in res and len(res["combo2_df"]):
        all_rows.extend(res["combo2_df"].to_dict("records"))
    all_summaries[k] = res["summary"]

result_df = pd.DataFrame(all_rows)
# keep 4 decimals for numeric columns
result_df = result_df.round(4)
mode_order = pd.Categorical(result_df["mode"], categories=["single","combo2","multi"], ordered=True)
result_df = result_df.assign(_mode_order=mode_order)
result_df.sort_values(["scenario", "_mode_order", "reconstructability"], ascending=[True, True, False], inplace=True)
result_df.drop(columns=["_mode_order"], inplace=True)
result_df


/var/folders/kc/c74q_vx95dx4__wm8dtz6q540000gn/T/ipykernel_22451/4149512713.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
/var/folders/kc/c74q_vx95dx4__wm8dtz6q540000gn/T/ipykernel_22451/4149512713.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
/var/folders/kc/c74q_vx95dx4__wm8dtz6q540000gn/T/ipykernel_22451/4149512713.py:28: FutureW

,scenario,name,run_mode,mode,source_set,n_events,expected_steps,expected_steps_set,n_expected_steps,expected_order,...,tagger_total_hits,tagger_missing_fields_events,tagger_missing_prefilter_events,tagger_used_prefilter_conditions,tagger_unusable_rules,observed_techniques,aligned_techniques,missing_expected_techniques,unexpected_observed_techniques,failure_keys
2,SC1,Stegano,single,single,azure_process,345,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,4,0,345,1,[],NaN,NaN,NaN,NaN,"[MISSING_DOWNLOAD, MISSING_EXFIL, MISSING_OUTB..."
1,SC1,Stegano,single,single,azure_conn,5746,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,0,5746,5746,1,[HEUR_AZURE_CONN_SKIPPED],NaN,NaN,NaN,NaN,"[MISSING_DOWNLOAD, MISSING_EXFIL, MISSING_INST..."
3,SC1,Stegano,single,single,azure_security,334,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,0,0,334,1,[],NaN,NaN,NaN,NaN,"[MISSING_DOWNLOAD, MISSING_EXFIL, MISSING_INST..."
4,SC1,Stegano,single,single,azure_port,1109,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,0,0,1109,1,[],NaN,NaN,NaN,NaN,"[MISSING_DOWNLOAD, MISSING_EXFIL, MISSING_INST..."
0,SC1,Stegano,multi,multi,azure_conn+azure_process+azure_security+azure_...,8534,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,4,5746,8534,1,[HEUR_AZURE_CONN_PARTIAL],[],[],"[T1001, T1001.002, T1003, T1003.006, T1005, T1...",[],[]
6,SC2,Starter,single,single,syslog,9732,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,11,0,9732,1,[],NaN,NaN,NaN,NaN,"[MISSING_DOWNLOAD, MISSING_OUTBOUND_CONN, PREF..."
5,SC2,Starter,multi,multi,azure_events+syslog,53978,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,101,0,53978,1,[],"[T1021.001, T1021.004, T1033, T1053.003, T1059...","[T1033, T1059.004, T1110, T1548.003]","[T1003, T1003.006, T1005, T1007, T1010, T1012,...","[T1021.001, T1021.004, T1053.003]",[]
7,SC2,Starter,combo2,multi,syslog+azure_events,53978,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,101,0,53978,1,[],NaN,NaN,NaN,NaN,"[MISSING_OUTBOUND_CONN, PREFILTER_UNUSABLE]"
10,SC3,Parallel,single,single,auth,2810,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,3,0,2810,1,[],NaN,NaN,NaN,NaN,"[MISSING_DOWNLOAD, MISSING_EXFIL, MISSING_OUTB..."
11,SC3,Parallel,single,single,suricata,65385,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]","[DOWNLOAD, EXFIL, INSTALL, OUTBOUND_CONN]",4,"[INSTALL, DOWNLOAD, OUTBOUND_CONN, EXFIL]",...,41667,0,65385,1,[],NaN,NaN,NaN,NaN,"[MISSING_DOWNLOAD, MISSING_EXFIL, MISSING_INST..."


In [11]:
result_df.to_csv("total_scenarios.csv", index=False)

### Evidence Report

In [12]:
import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import numpy as np


In [13]:
# --------- helpers: canonical field picking ---------
def _pick_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _json_default(o):
    import numpy as np
    import pandas as pd

    # numpy scalars
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)

    # numpy arrays
    if isinstance(o, (np.ndarray,)):
        return o.tolist()

    # pandas timestamps / timedeltas
    if isinstance(o, (pd.Timestamp,)):
        return o.isoformat()
    if isinstance(o, (pd.Timedelta,)):
        return str(o)

    # last resort: let json raise (keeps you honest)
    raise TypeError(f"Object of type {type(o).__name__} is not JSON serializable")

def _mk_excerpt(row: pd.Series, text_cols: List[str], max_len: int = 240) -> str:
    parts = []
    for c in text_cols:
        if c in row and pd.notna(row[c]):
            s = str(row[c]).strip()
            if s:
                parts.append(s)
    txt = " | ".join(parts)
    txt = re.sub(r"\s+", " ", txt)
    return (txt[:max_len] + "…") if len(txt) > max_len else txt

def _canonicalize_events(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with a canonical subset of columns (if present)."""
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()

    # Common timestamp columns (your pipeline already normalizes to 'ts' when possible)
    ts_col = _pick_col(out, ["ts", "@timestamp", "timestamp", "TimeGenerated", "time"])
    if ts_col and ts_col != "ts":
        out["ts"] = out[ts_col]

    # ensure datetime (naive)
    if "ts" in out.columns:
        out["ts"] = pd.to_datetime(out["ts"], errors="coerce", utc=True).dt.tz_convert(None)

    # source column
    src_col = _pick_col(out, ["source", "Source", "dataset", "Type", "log"])
    if src_col and src_col != "source":
        out["source"] = out[src_col]

    # rule / label columns (best-effort; depends on your tagger implementation)
    rule_col = _pick_col(out, ["rule_id", "rule", "tag_rule", "tagger_rule", "matched_rule"])
    if rule_col and rule_col != "rule_id":
        out["rule_id"] = out[rule_col]

    # handy observables
    # (these may or may not exist depending on parser)
    for canon, cands in {
        "exe": ["exe", "process", "ProcessName", "Image", "process_name"],
        "cmdline": ["cmdline", "CommandLine", "command", "proctitle_decoded", "proctitle", "SyslogMessage", "message", "raw", "event"],
        "user": ["user", "User", "AccountName", "username"],
        "pid": ["pid", "ProcessId", "process_id"],
        "ppid": ["ppid", "ParentProcessId", "parent_pid"],
        "src_ip": ["src_ip", "SourceIP", "id.orig_h", "saddr"],
        "src_port": ["src_port", "SourcePort", "id.orig_p", "sport"],
        "dst_ip": ["dst_ip", "DestinationIP", "dest_ip", "id.resp_h", "daddr"],
        "dst_port": ["dst_port", "DestinationPort", "dest_port", "id.resp_p", "dport"],
        "proto": ["proto", "Protocol", "transport", "service"],
        "bytes_in": ["bytes_in", "BytesReceived", "resp_bytes", "bytes_received"],
        "bytes_out": ["bytes_out", "BytesSent", "orig_bytes", "bytes_sent"],
        "cwd": ["cwd", "CurrentDirectory", "process_working_directory", "WorkingDirectory"],
    }.items():
        col = _pick_col(out, cands)
        if col and col != canon:
            out[canon] = out[col]

    # build an excerpt field for paper-friendly evidence
    text_cols = [c for c in ["cmdline", "command", "SyslogMessage", "message", "raw", "event", "proctitle_decoded", "exe"] if c in out.columns]
    if "excerpt" not in out.columns:
        out["excerpt"] = out.apply(lambda r: _mk_excerpt(r, text_cols), axis=1)

    return out


In [14]:
# --------- core export: per-step evidence bundle ---------
def build_step_evidence(
    tagged: pd.DataFrame,
    step: str,
    top_k: int = 5,
    pad_seconds: int = 60,
) -> Dict:
    """Summarize evidence for one step: time window, sources, representative events."""
    df = tagged.copy()
    if "step" not in df.columns:
        return {"step": step, "present": False, "reason": "NO_STEP_COLUMN"}

    df = df[df["step"] == step].copy()
    if len(df) == 0:
        return {"step": step, "present": False, "reason": "NO_EVENTS_FOR_STEP"}

    df = _canonicalize_events(df)

    # time window
    t0 = df["ts"].min() if "ts" in df.columns else None
    t1 = df["ts"].max() if "ts" in df.columns else None

    # padded window (for aligning in the narrative)
    if pd.notna(t0) and pd.notna(t1):
        w0 = (t0 - pd.Timedelta(seconds=pad_seconds)).to_pydatetime()
        w1 = (t1 + pd.Timedelta(seconds=pad_seconds)).to_pydatetime()
        window = [w0.isoformat(timespec="seconds"), w1.isoformat(timespec="seconds")]
    else:
        window = [None, None]

    # representative events: prefer diversity by source, then chronological
    df = df.sort_values(["ts"] if "ts" in df.columns else df.columns.tolist())
    ev = []

    # 1) take up to 1 per source
    if "source" in df.columns:
        for src, g in df.groupby("source", sort=False):
            row = g.iloc[0]
            ev.append(row)
            if len(ev) >= top_k:
                break

    # 2) fill remaining chronologically
    if len(ev) < top_k:
        taken_idx = set([getattr(x, "name", None) for x in ev])
        for _, row in df.iterrows():
            if len(ev) >= top_k:
                break
            if row.name in taken_idx:
                continue
            ev.append(row)

    # serialize
    def _ser(row: pd.Series) -> Dict:
        out = {}
        for k in ["ts", "source", "exe", "cmdline", "user", "pid", "ppid",
                  "src_ip", "src_port", "dst_ip", "dst_port", "proto",
                  "bytes_in", "bytes_out", "cwd", "rule_id", "excerpt"]:
            if k in row and pd.notna(row[k]):
                v = row[k]
                out[k] = v.isoformat(timespec="seconds") if isinstance(v, (pd.Timestamp,)) else v
        return out

    sources = sorted(set(df["source"].dropna().astype(str).tolist())) if "source" in df.columns else []
    return {
        "step": step,
        "present": True,
        "time_window": window,
        "n_events": int(len(df)),
        "sources": sources,
        "evidence": [_ser(x) for x in ev],
    }


def export_evidence_bundle(
    scenario_id: str,
    scenario_name: str,
    run_out: Dict,
    out_dir: Path | str = "case_evidence",
    top_k: int = 5,
) -> Path:
    """
    Export:
      - <scenario>_evidence.json  (per-step evidence bundle)
      - <scenario>_step_events/<STEP>.csv  (all events for each step)
      - <scenario>_timeline.csv  (step ordering by first timestamp)
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    tagged = run_out.get("tagged", pd.DataFrame())
    recon = run_out.get("recon", {}) or {}
    chain_steps = recon.get("chain_steps") or run_out.get("chain") or []
    observed_steps = recon.get("observed_steps") or (sorted(tagged["step"].dropna().unique()) if "step" in tagged.columns else [])

    # prefer chain order if available; else order by first timestamp
    steps = chain_steps if chain_steps else observed_steps

    # build evidence
    bundle = {
        "scenario": scenario_id,
        "name": scenario_name,
        "reconstruction": {
            "expected_steps": recon.get("expected_steps", []),
            "observed_steps": recon.get("observed_steps", []),
            "chain_steps": chain_steps,
            "step_recall": recon.get("step_recall"),
            "step_precision": recon.get("step_precision"),
            "chain_recall": recon.get("chain_recall"),
            "chain_precision": recon.get("chain_precision"),
            "reconstructability": recon.get("reconstructability"),
            "reconstructability_mode": recon.get("reconstructability_mode"),
            "n_events": recon.get("n_events"),
        },
        "steps": [],
    }

    # export per-step
    step_events_dir = out_dir / f"{scenario_id}_step_events"
    step_events_dir.mkdir(parents=True, exist_ok=True)

    tagged_can = _canonicalize_events(tagged) if len(tagged) else pd.DataFrame()

    for st in steps:
        ev = build_step_evidence(tagged, st, top_k=top_k)
        bundle["steps"].append(ev)

        # write full events for this step
        if len(tagged_can) and "step" in tagged_can.columns:
            sdf = tagged_can[tagged_can["step"] == st].copy()
            if len(sdf):
                # keep a compact but useful column set
                keep = [c for c in [
                    "ts","source","step","exe","cmdline","user","pid","ppid",
                    "src_ip","src_port","dst_ip","dst_port","proto",
                    "bytes_in","bytes_out","cwd","rule_id","excerpt","source_file"
                ] if c in sdf.columns]
                sdf[keep].to_csv(step_events_dir / f"{st}.csv", index=False)

    # timeline table (for Fig. timeline or case narrative)
    timeline_cols = ["step", "t_first", "t_last", "n_events", "sources"]
    timeline_rows = []

    if len(tagged_can) and {"step", "ts"}.issubset(tagged_can.columns):
        tmp = tagged_can.dropna(subset=["ts"]).copy()
        if len(tmp):
            for st, g in tmp.groupby("step", dropna=True):
                if pd.isna(st):
                    continue
                timeline_rows.append({
                    "step": st,
                    "t_first": g["ts"].min(),
                    "t_last": g["ts"].max(),
                    "n_events": int(len(g)),
                    "sources": ";".join(sorted(set(g["source"].dropna().astype(str)))) if "source" in g.columns else ""
                })

    tl = pd.DataFrame(timeline_rows, columns=timeline_cols)
    if len(tl):
        tl = tl.sort_values("t_first")

    tl.to_csv(out_dir / f"{scenario_id}_timeline.csv", index=False)


    # write JSON bundle
    out_path = out_dir / f"{scenario_id}_evidence.json"
    out_path.write_text(
        json.dumps(bundle, indent=2, ensure_ascii=False, default=_json_default),
        encoding="utf-8"
    )
    
    return out_path


def export_single_vs_multi_matrix(res: Dict, out_dir: Path | str = "case_evidence") -> Path:
    """
    Export a compact matrix for the case study:
      - which steps each single-source can see vs multi
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    multi_steps = set((res.get("multi", {}).get("recon", {}) or {}).get("observed_steps") or [])
    rows = []
    for src, out in (res.get("single") or {}).items():
        s_steps = set((out.get("recon", {}) or {}).get("observed_steps") or [])
        rows.append({
            "source": src,
            "n_steps": len(s_steps),
            "steps": ";".join(sorted(s_steps)),
            "missing_vs_multi": ";".join(sorted(multi_steps - s_steps)),
            "extra_vs_multi": ";".join(sorted(s_steps - multi_steps)),
        })

    df = pd.DataFrame(rows).sort_values(["n_steps", "source"], ascending=[False, True])
    p = out_dir / f"{res['summary']['scenario']}_single_vs_multi_steps.csv"
    df.to_csv(p, index=False)
    return p

In [15]:
from pathlib import Path

OUTDIR = Path("case_evidence")

# SC1
res_sc1 = analyze_scenario("sc1", LOG_ROOT)
p1 = export_evidence_bundle(
    scenario_id=res_sc1["summary"]["scenario"],
    scenario_name=res_sc1["summary"]["name"],
    run_out=res_sc1["multi"],
    out_dir=OUTDIR,
    top_k=5
)
p1b = export_single_vs_multi_matrix(res_sc1, out_dir=OUTDIR)

print("SC1 evidence:", p1)
print("SC1 single-vs-multi:", p1b)

# SC4
res_sc4 = analyze_scenario("sc4", LOG_ROOT)
p4 = export_evidence_bundle(
    scenario_id=res_sc4["summary"]["scenario"],
    scenario_name=res_sc4["summary"]["name"],
    run_out=res_sc4["multi"],
    out_dir=OUTDIR,
    top_k=5
)
p4b = export_single_vs_multi_matrix(res_sc4, out_dir=OUTDIR)

print("SC4 evidence:", p4)
print("SC4 single-vs-multi:", p4b)


SC1 evidence: case_evidence/SC1_evidence.json
SC1 single-vs-multi: case_evidence/SC1_single_vs_multi_steps.csv


/var/folders/kc/c74q_vx95dx4__wm8dtz6q540000gn/T/ipykernel_22451/4149512713.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


SC4 evidence: case_evidence/SC4_evidence.json
SC4 single-vs-multi: case_evidence/SC4_single_vs_multi_steps.csv
